In order to generate the partitioning traces, run the following:

```
python3 compute_partitioning.py -j8
```

In [ ]:
# Loop coverage helpers and daughter classification
N_PARENT = 54338


def _beads_on_shorter_arc_circular(p1, p2, N):
    """Return set of bead indices on the shorter arc between p1 and p2 on a circle 0..N-1."""
    if N <= 0:
        return set()
    p1 = max(0, min(p1, N - 1))
    p2 = max(0, min(p2, N - 1))
    p_lo, p_hi = min(p1, p2), max(p1, p2)
    forward = p_hi - p_lo
    backward = N - forward
    if forward <= backward:
        return set(range(p_lo, p_hi + 1))
    else:
        return set(range(p_hi, N)) | set(range(0, p_lo + 1))


def _on_left_daughter(p, fork1, fork2, N_parent=N_PARENT):
    """True if bead index p lies on the left-daughter segment between forks on the parent circle.

    Geometry (parent circle indices 0..N_parent-1):
      - Before replication: fork1 == fork2, no left daughter yet.
      - During replication: left daughter is the arc from fork1 to fork2 following
        increasing index and wrapping around 0 if needed.
      - After replication: forks are separated so the left arc covers the entire circle.
    """
    if fork1 == fork2:
        return False
    if p < 0 or p >= N_parent:
        return False
    if fork1 <= fork2:
        return fork1 <= p <= fork2
    # fork1 > fork2: left arc wraps around 0
    return p >= fork1 or p <= fork2


def _classify_loop_by_daughter(p1, p2, fork1, fork2, N_parent=N_PARENT):
    """Classify a loop by daughter: 'left', 'right', or 'other'.

    - Right daughter: both anchors on the replicated right strand (>= N_parent).
    - Left daughter: both anchors on the parent circle and between forks (left arc).
    - Other: any loop that doesn't satisfy those conditions (e.g. crossing regions).
    """
    if p1 >= N_parent and p2 >= N_parent:
        return "right"
    if p1 < N_parent and p2 < N_parent:
        if _on_left_daughter(p1, fork1, fork2, N_parent) and _on_left_daughter(p2, fork1, fork2, N_parent):
            return "left"
    return "other"


def coverage_sum_of_spans(loops1, loops2, N_parent, current_max_bead, num_smc, fork1, fork2):
    """Method 1: Total coverage = sum of each loop's span on left/right daughters.

    Loops are assigned to daughters using _classify_loop_by_daughter; loops classified
    as 'other' are ignored for coverage.
    """
    left_cov = 0
    right_cov = 0
    right_region_size = max(0, current_max_bead - N_parent + 1)
    is_right_circular = current_max_bead >= 2 * N_parent - 1

    n = min(num_smc, len(loops1), len(loops2))
    for i in range(n):
        p1, p2 = loops1[i], loops2[i]
        if p1 == 0 and p2 == 0:
            continue
        cls = _classify_loop_by_daughter(p1, p2, fork1, fork2, N_parent)
        if cls == "left":
            # Work on parent circle 0..N_parent-1
            p1c = p1 % N_parent
            p2c = p2 % N_parent
            span = abs(p2c - p1c)
            span = min(span, N_parent - span)
            left_cov += span
        elif cls == "right" and right_region_size > 0:
            lo = min(p1, p2) - N_parent
            hi = max(p1, p2) - N_parent
            lo = max(0, min(lo, right_region_size - 1))
            hi = max(0, min(hi, right_region_size - 1))
            if hi < lo:
                lo, hi = hi, lo
            span = hi - lo
            if is_right_circular:
                span = min(span, right_region_size - span)
            right_cov += span
    return left_cov, right_cov, left_cov + right_cov


def coverage_unique_sites(loops1, loops2, N_parent, current_max_bead, num_smc, fork1, fork2):
    """Method 2: Unique coverage = number of distinct sites covered on each daughter.

    Uses boolean arrays per daughter; loops are first classified by daughter.
    """
    left_covered = [False] * N_parent
    right_region_size = max(0, current_max_bead - N_parent + 1)
    right_covered = [False] * right_region_size if right_region_size > 0 else []
    is_right_circular = current_max_bead >= 2 * N_parent - 1

    n = min(num_smc, len(loops1), len(loops2))
    for i in range(n):
        p1, p2 = loops1[i], loops2[i]
        if p1 == 0 and p2 == 0:
            continue
        cls = _classify_loop_by_daughter(p1, p2, fork1, fork2, N_parent)
        if cls == "left":
            p1c = p1 % N_parent
            p2c = p2 % N_parent
            for idx in _beads_on_shorter_arc_circular(p1c, p2c, N_parent):
                if _on_left_daughter(idx, fork1, fork2, N_parent):
                    left_covered[idx] = True
        elif cls == "right" and right_region_size > 0:
            lo = min(p1, p2) - N_parent
            hi = max(p1, p2) - N_parent
            lo = max(0, min(lo, right_region_size - 1))
            hi = max(0, min(hi, right_region_size - 1))
            if hi < lo:
                lo, hi = hi, lo
            if is_right_circular:
                idxs = _beads_on_shorter_arc_circular(lo, hi, right_region_size)
            else:
                idxs = range(lo, hi + 1)
            for idx in idxs:
                if 0 <= idx < right_region_size:
                    right_covered[idx] = True

    left_cov = sum(1 for x in left_covered if x)
    right_cov = sum(1 for x in right_covered if x)
    return left_cov, right_cov, left_cov + right_cov


In [ ]:
# Override coverage functions to accept optional fork positions

def coverage_sum_of_spans(loops1, loops2, N_parent, current_max_bead, num_smc, fork1=None, fork2=None):
    """Method 1: Total coverage = sum of each loop's span on left/right daughters.

    If fork1/fork2 are provided, loops are assigned using _classify_loop_by_daughter.
    Otherwise, a simpler rule is used: anchors < N_parent -> left, >= N_parent -> right.
    """
    left_cov = 0
    right_cov = 0
    right_region_size = max(0, current_max_bead - N_parent + 1)
    is_right_circular = current_max_bead >= 2 * N_parent - 1

    n = min(num_smc, len(loops1), len(loops2))
    for i in range(n):
        p1, p2 = loops1[i], loops2[i]
        if p1 == 0 and p2 == 0:
            continue
        if fork1 is not None and fork2 is not None:
            cls = _classify_loop_by_daughter(p1, p2, fork1, fork2, N_parent)
        else:
            if p1 >= N_parent and p2 >= N_parent:
                cls = "right"
            elif p1 < N_parent and p2 < N_parent:
                cls = "left"
            else:
                cls = "other"
        if cls == "left":
            p1c = p1 % N_parent
            p2c = p2 % N_parent
            span = abs(p2c - p1c)
            span = min(span, N_parent - span)
            left_cov += span
        elif cls == "right" and right_region_size > 0:
            lo = min(p1, p2) - N_parent
            hi = max(p1, p2) - N_parent
            lo = max(0, min(lo, right_region_size - 1))
            hi = max(0, min(hi, right_region_size - 1))
            if hi < lo:
                lo, hi = hi, lo
            span = hi - lo
            if is_right_circular:
                span = min(span, right_region_size - span)
            right_cov += span
    return left_cov, right_cov, left_cov + right_cov


def coverage_unique_sites(loops1, loops2, N_parent, current_max_bead, num_smc, fork1=None, fork2=None):
    """Method 2: Unique coverage = number of distinct sites covered on each daughter.

    Uses boolean arrays per daughter; loops are first classified by daughter when
    fork positions are known; otherwise falls back to a simple index-based rule.
    """
    left_covered = [False] * N_parent
    right_region_size = max(0, current_max_bead - N_parent + 1)
    right_covered = [False] * right_region_size if right_region_size > 0 else []
    is_right_circular = current_max_bead >= 2 * N_parent - 1

    n = min(num_smc, len(loops1), len(loops2))
    for i in range(n):
        p1, p2 = loops1[i], loops2[i]
        if p1 == 0 and p2 == 0:
            continue
        if fork1 is not None and fork2 is not None:
            cls = _classify_loop_by_daughter(p1, p2, fork1, fork2, N_parent)
        else:
            if p1 >= N_parent and p2 >= N_parent:
                cls = "right"
            elif p1 < N_parent and p2 < N_parent:
                cls = "left"
            else:
                cls = "other"
        if cls == "left":
            p1c = p1 % N_parent
            p2c = p2 % N_parent
            for idx in _beads_on_shorter_arc_circular(p1c, p2c, N_parent):
                if fork1 is None or fork2 is None or _on_left_daughter(idx, fork1, fork2, N_parent):
                    left_covered[idx] = True
        elif cls == "right" and right_region_size > 0:
            lo = min(p1, p2) - N_parent
            hi = max(p1, p2) - N_parent
            lo = max(0, min(lo, right_region_size - 1))
            hi = max(0, min(hi, right_region_size - 1))
            if hi < lo:
                lo, hi = hi, lo
            if is_right_circular:
                idxs = _beads_on_shorter_arc_circular(lo, hi, right_region_size)
            else:
                idxs = range(lo, hi + 1)
            for idx in idxs:
                if 0 <= idx < right_region_size:
                    right_covered[idx] = True

    left_cov = sum(1 for x in left_covered if x)
    right_cov = sum(1 for x in right_covered if x)
    return left_cov, right_cov, left_cov + right_cov


In [ ]:
# Partitioning plot: load from partitioning/*.txt and plot. Use plot_partitioning_trajectories() in later cells for different date/replicate sets.
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

def load_partitioning_file(filepath):
    """Load minute and partitioning arrays from one replicate .txt file."""
    minutes, part = [], []
    with open(filepath) as f:
        for line in f:
            line = line.rstrip()
            if line.startswith("#") or not line:
                continue
            parts = line.split("\t")
            if len(parts) >= 2:
                minutes.append(int(parts[0]))
                part.append(float(parts[1]))
    return (np.array(minutes), np.array(part)) if minutes else (np.array([]), np.array([]))

def plot_partitioning_trajectories(dates=None, replicates=None, partitioning_dir=None, base_dir=None, ax=None, title="Daughter chromosome partitioning vs time", figsize=(8, 5), labels=None, ylim=None, xlim=None):
    """
    Load partitioning .txt files and plot one trace per replicate.
    labels: optional dict mapping replicate name -> legend label (e.g. {"Feb08_1": "1× speed"}); missing keys use replicate name.
    Trace and legend order follow the order of labels (if provided) or the order of replicates.
    ylim: optional (ymin, ymax) for y-axis limits; default (0.8, 1.0). Use (0.7, 1.05) or (None, 1.05) to set only max.
    """
    base_dir = Path(base_dir) if base_dir is not None else Path(".")
    part_dir = Path(partitioning_dir) if partitioning_dir is not None else base_dir / "partitioning"
    if replicates is None:
        if not dates:
            raise ValueError("Provide either dates or replicates")
        replicates = sorted(
            f.stem for f in part_dir.glob("*.txt")
            if any(f.stem.startswith(d) for d in dates)
        )
    if not replicates:
        raise FileNotFoundError(f"No partitioning files found for dates {dates} in {part_dir}")
    series = {}
    for rep in replicates:
        path = part_dir / f"{rep}.txt"
        if path.exists():
            minutes, part = load_partitioning_file(path)
            if len(minutes) > 0:
                series[rep] = (minutes, part)
    if not series:
        raise FileNotFoundError(f"No data loaded for {replicates} in {part_dir}")
    # Plot/legend order: order of labels if provided, else order of replicates, else sorted
    if labels is not None:
        plot_order = [rep for rep in labels if rep in series]
    else:
        plot_order = []
    if not plot_order:
        plot_order = [rep for rep in replicates if rep in series]
    if not plot_order:
        plot_order = sorted(series.keys())
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
        created_fig = True
    else:
        fig, created_fig = None, False
    colors = plt.cm.tab10(np.linspace(0, 1, max(10, len(series))))
    for k, rep in enumerate(plot_order):
        minutes, part = series[rep]
        order = np.argsort(minutes)
        trace_label = labels.get(rep, rep) if labels else rep
        ax.plot(minutes[order], part[order], label=trace_label, color=colors[k % len(colors)])
    ax.set_xlabel("Time (min)")
    ax.set_ylabel("Partitioning")
    default_ymin, default_ymax = 0.8, 1.0
    default_xmin, default_xmax = 0, 100
    if ylim is not None and xlim is None:
        ymin = ylim[0] if ylim[0] is not None else default_ymin
        ymax = ylim[1] if len(ylim) > 1 and ylim[1] is not None else default_ymax
    elif ylim is not None and xlim is not None:
        ymin = ylim[0] if ylim[0] is not None else default_ymin
        ymax = ylim[1] if len(ylim) > 1 and ylim[1] is not None else default_ymax
        xmin = xlim[0] if xlim[0] is not None else default_xmin
        xmax = xlim[1] if len(xlim) > 1 and xlim[1] is not None else default_xmax
    else:
        ymin, ymax = default_ymin, default_ymax
        xmin, xmax = default_xmin, default_xmax
    ax.set_ylim(ymin, ymax)
    ax.set_xlim(xmin, xmax)
    ax.legend()
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    if created_fig:
        plt.tight_layout()
    return (fig, ax) if created_fig else (None, ax)

base_dir = Path("/raid/amaytin/protein_science/runs/")
#plot_partitioning_trajectories(dates=["Jan22", "Jan26", "Jan27"], base_dir=base_dir)
#plt.show()
#plt.show()

In [ ]:
def plot_partitioning_trajectories(
    dates=None,
    replicates=None,
    partitioning_dir=None,
    base_dir=None,
    ax=None,
    title="Daughter chromosome partitioning vs time",
    figsize=(8, 5),
    labels=None,
    ylim=None,
    xlim=None,
    frames_per_minute=15.0,
    xlabel_fontsize=None,
    ylabel_fontsize=None,
    title_fontsize=None,
    tick_labelsize=None,
    legend_fontsize=None,
):
    """Load partitioning .txt files and plot one trace per replicate.

    labels: optional dict mapping replicate name -> legend label (e.g. {"Feb08_1": "1× speed"});
        missing keys use replicate name.
    Trace and legend order follow the order of labels (if provided) or the order of replicates.
    ylim: optional (ymin, ymax) for y-axis limits; default (0.8, 1.0). Use (0.7, 1.05) or (None, 1.05) to set only max.
    frames_per_minute: conversion factor from frames to minutes for the x-axis (default 15).
    """
    base_dir = Path(base_dir) if base_dir is not None else Path(".")
    part_dir = Path(partitioning_dir) if partitioning_dir is not None else base_dir / "partitioning"
    if replicates is None:
        if not dates:
            raise ValueError("Provide either dates or replicates")
        replicates = sorted(
            f.stem for f in part_dir.glob("*.txt")
            if any(f.stem.startswith(d) for d in dates)
        )
    if not replicates:
        raise FileNotFoundError(f"No partitioning files found for dates {dates} in {part_dir}")
    series = {}
    for rep in replicates:
        path = part_dir / f"{rep}.txt"
        if path.exists():
            minutes, part = load_partitioning_file(path)
            if len(minutes) > 0:
                series[rep] = (minutes, part)
    if not series:
        raise FileNotFoundError(f"No data loaded for {replicates} in {part_dir}")
    # Plot/legend order: order of labels if provided, else order of replicates, else sorted
    if labels is not None:
        plot_order = [rep for rep in labels if rep in series]
    else:
        plot_order = []
    if not plot_order:
        plot_order = [rep for rep in replicates if rep in series]
    if not plot_order:
        plot_order = sorted(series.keys())
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
        created_fig = True
    else:
        fig, created_fig = None, False
    colors = plt.cm.tab10(np.linspace(0, 1, max(10, len(series))))
    for k, rep in enumerate(plot_order):
        minutes, part = series[rep]
        order = np.argsort(minutes)
        time_min = minutes[order] / float(frames_per_minute)
        trace_label = labels.get(rep, rep) if labels else rep
        ax.plot(time_min, part[order], label=trace_label, color=colors[k % len(colors)])
    # Determine default axis limits from data (after conversion to minutes)
    all_minutes = [series[rep][0] for rep in plot_order if len(series[rep][0]) > 0]
    if all_minutes:
        max_minute = max(m.max() for m in all_minutes)
    else:
        max_minute = 0
    default_ymin, default_ymax = 0.8, 1.0
    default_xmin, default_xmax = 0.0, (max_minute / float(frames_per_minute)) if max_minute > 0 else 1.0
    # Apply axis limits, allowing independent control of x and y
    if ylim is not None:
        ymin = ylim[0] if ylim[0] is not None else default_ymin
        ymax = ylim[1] if len(ylim) > 1 and ylim[1] is not None else default_ymax
    else:
        ymin, ymax = default_ymin, default_ymax
    if xlim is not None:
        xmin = xlim[0] if xlim[0] is not None else default_xmin
        xmax = xlim[1] if len(xlim) > 1 and xlim[1] is not None else default_xmax
    else:
        xmin, xmax = default_xmin, default_xmax
    ax.set_ylim(ymin, ymax)
    ax.set_xlim(xmin, xmax)
    ax.set_xlabel("Time (min)", fontsize=xlabel_fontsize)
    ax.set_ylabel("Partitioning", fontsize=ylabel_fontsize)
    if tick_labelsize is not None:
        ax.tick_params(axis="both", labelsize=tick_labelsize)
    legend = ax.legend()
    if legend is not None and legend_fontsize is not None:
        for text in legend.get_texts():
            text.set_fontsize(legend_fontsize)
    ax.set_title(title, fontsize=title_fontsize)
    ax.grid(True, alpha=0.3)
    if created_fig:
        plt.tight_layout()
    return (fig, ax) if created_fig else (None, ax)


In [ ]:
def plot_loops(replicate, N_parent=N_PARENT, use_concatenated_loops=True, frames_per_minute=30):
    """Plot loop coverages and counts vs time for a given replicate.

    Uses fork-aware daughter classification for both coverage and loop counts.
    By default reads from the concatenated loops_<replicate>.txt file.
    """
    import os
    import numpy as np
    import matplotlib.pyplot as plt

    if use_concatenated_loops:
        concat_path = f"../runs/{replicate}/data/loops/loops_{replicate}.txt"

        def _parse_concatenated_loops(path, N_parent_local=N_parent):
            """Parse concatenated loops_*.txt into snapshots.

            Returns list of (loops1, loops2, fork1, fork2) for each snapshot.
            The file format repeats blocks of:
              Number of loops: X
              Replication forks: L, R
              <X lines of loop data>
            """
            snapshots_local = []
            with open(path, "r") as f:
                lines = [ln.strip() for ln in f if ln.strip()]
            i = 0
            n = len(lines)
            while i < n:
                line = lines[i]
                if not line.startswith("Number of loops"):
                    i += 1
                    continue
                # Number of loops
                try:
                    _, val = line.split(":", 1)
                    n_loops = int(val)
                except Exception:
                    n_loops = 0
                # Next line: forks
                i += 1
                if i >= n:
                    break
                fork_line = lines[i]
                try:
                    _, vals = fork_line.split(":", 1)
                    fork_parts = vals.strip().split(",")
                    fork1 = int(fork_parts[0])
                    fork2 = int(fork_parts[1])
                except Exception:
                    fork1 = fork2 = 0
                # Following lines: loop anchors
                loops1, loops2 = [], []
                i += 1
                loops_read = 0
                while i < n and loops_read < n_loops and not lines[i].startswith("Number of loops"):
                    cols = lines[i].split()
                    if len(cols) >= 2:
                        try:
                            p1 = int(cols[0])
                            p2 = int(cols[1])
                            loops1.append(p1)
                            loops2.append(p2)
                            loops_read += 1
                        except ValueError:
                            pass
                    i += 1
                snapshots_local.append((loops1, loops2, fork1, fork2))
            return snapshots_local

        snapshots = _parse_concatenated_loops(concat_path)
        loop_times = np.arange(len(snapshots)) / float(frames_per_minute)

        # Coverage and loop counts from concatenated file
        left_sum_cat, right_sum_cat, total_sum_cat = [], [], []
        left_uniq_cat, right_uniq_cat, total_uniq_cat = [], [], []
        left_loops_cat, right_loops_cat, total_loops_cat = [], [], []

        for (loops1, loops2, fork1, fork2) in snapshots:
            n_loops = len(loops1)
            if n_loops == 0:
                left_sum_cat.append(0)
                right_sum_cat.append(0)
                total_sum_cat.append(0)
                left_uniq_cat.append(0)
                right_uniq_cat.append(0)
                total_uniq_cat.append(0)
                left_loops_cat.append(0)
                right_loops_cat.append(0)
                total_loops_cat.append(0)
                continue
            # current_max_bead from fork separation
            right_len = abs(fork2 - fork1)
            current_max_bead = N_parent + right_len - 1 if right_len > 0 else N_parent - 1

            # Fork-aware coverage on left/right daughters
            l1, r1, t1 = coverage_sum_of_spans(loops1, loops2, N_parent, current_max_bead, n_loops, fork1, fork2)
            left_sum_cat.append(l1)
            right_sum_cat.append(r1)
            total_sum_cat.append(t1)

            l2, r2, t2 = coverage_unique_sites(loops1, loops2, N_parent, current_max_bead, n_loops, fork1, fork2)
            left_uniq_cat.append(l2)
            right_uniq_cat.append(r2)
            total_uniq_cat.append(t2)

            # Loop counts per daughter using same classification
            left_cnt = right_cnt = 0
            for p1, p2 in zip(loops1, loops2):
                cls = _classify_loop_by_daughter(p1, p2, fork1, fork2)
                if cls == "left":
                    left_cnt += 1
                elif cls == "right":
                    right_cnt += 1
            left_loops_cat.append(left_cnt)
            right_loops_cat.append(right_cnt)
            total_loops_cat.append(n_loops)

        # Plot coverage and loop counts
        fig, axes = plt.subplots(4, 1, figsize=(10, 11), sharex=True)
        ax0, ax1, ax2, ax3 = axes

        ax0.set_title(f"Coverage (sum of spans) — {replicate}")
        ax0.plot(loop_times, left_sum_cat, label="Left daughter", color="C0", alpha=0.9)
        ax0.plot(loop_times, right_sum_cat, label="Right daughter", color="C1", alpha=0.9)
        ax0.plot(loop_times, total_sum_cat, label="Total", color="gray", linestyle="--", alpha=0.7)
        ax0.set_ylabel("Loop coverage (beads)")
        ax0.legend()
        ax0.grid(True, alpha=0.3)

        ax1.set_title("Coverage (unique sites)")
        ax1.plot(loop_times, left_uniq_cat, label="Left daughter", color="C0", alpha=0.9)
        ax1.plot(loop_times, right_uniq_cat, label="Right daughter", color="C1", alpha=0.9)
        ax1.plot(loop_times, total_uniq_cat, label="Total", color="gray", linestyle="--", alpha=0.7)
        ax1.set_ylabel("Unique covered sites (beads)")
        ax1.legend()
        ax1.grid(True, alpha=0.3)

        ax2.set_title("Coverage overlaps (sum - unique)")
        ax2.plot(loop_times, np.array(left_sum_cat) - np.array(left_uniq_cat), label="Left daughter", color="C0", alpha=0.9)
        ax2.plot(loop_times, np.array(right_sum_cat) - np.array(right_uniq_cat), label="Right daughter", color="C1", alpha=0.9)
        ax2.plot(loop_times, np.array(total_sum_cat) - np.array(total_uniq_cat), label="Total", color="gray", linestyle="--", alpha=0.7)
        ax2.set_ylabel("Overlaps (beads)")
        ax2.legend()
        ax2.grid(True, alpha=0.3)

        ax3.set_title("Loop counts per daughter")
        ax3.plot(loop_times, left_loops_cat, label="Left daughter loops", color="C0")
        ax3.plot(loop_times, right_loops_cat, label="Right daughter loops", color="C1")
        ax3.plot(loop_times, total_loops_cat, label="Total loops", color="gray", linestyle="--")
        ax3.set_xlabel("Time (min)")
        ax3.set_ylabel("Number of loops")
        ax3.legend()
        ax3.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

In [ ]:
def plot_loops_multiple(
    replicates,
    N_parent=N_PARENT,
    use_concatenated_loops=True,
    frames_per_minute=30,
    base_color_map=None,
    figsize=(10, 6),
):
    """Plot loop coverage and loop counts vs time for multiple replicates on shared axes.

    For each replicate, uses the same fork-aware classification as `plot_loops` on the
    concatenated `loops_<replicate>.txt` file.

    Coverage subplot:
      - Plots *total* coverage traces per replicate:
          - sum-of-spans coverage (lower opacity)
          - unique-sites coverage (higher opacity)
        so for R replicates, there are 2R traces.
    Loop-count subplot:
      - Plots left-daughter, right-daughter, and total loop counts for each replicate
        overlaid on a single axis (3R traces).
    """
    import numpy as np
    import matplotlib.pyplot as plt
    import matplotlib.colors as mcolors

    replicates = list(replicates)
    if not replicates:
        raise ValueError("Provide at least one replicate")

    # Colors: one base color per replicate
    if base_color_map is None:
        cmap = plt.cm.tab10
        colors = [cmap(i % 10) for i in range(len(replicates))]
    else:
        colors = [base_color_map(rep) for rep in replicates]

    # Prepare per-replicate time series
    series = {}
    for rep in replicates:
        if not use_concatenated_loops:
            raise NotImplementedError("plot_loops_multiple currently supports only use_concatenated_loops=True")
        concat_path = f"../runs/{rep}/data/loops/loops_{rep}.txt"

        # Reuse the parser logic from plot_loops
        def _parse_concatenated_loops(path, N_parent_local=N_parent):
            snapshots_local = []
            with open(path, "r") as f:
                lines = [ln.strip() for ln in f if ln.strip()]
            i = 0
            n = len(lines)
            while i < n:
                line = lines[i]
                if not line.startswith("Number of loops"):
                    i += 1
                    continue
                try:
                    _, val = line.split(":", 1)
                    n_loops = int(val)
                except Exception:
                    n_loops = 0
                i += 1
                if i >= n:
                    break
                fork_line = lines[i]
                try:
                    _, vals = fork_line.split(":", 1)
                    fork_parts = vals.strip().split(",")
                    fork1 = int(fork_parts[0])
                    fork2 = int(fork_parts[1])
                except Exception:
                    fork1 = fork2 = 0
                loops1, loops2 = [], []
                i += 1
                loops_read = 0
                while i < n and loops_read < n_loops and not lines[i].startswith("Number of loops"):
                    cols = lines[i].split()
                    if len(cols) >= 2:
                        try:
                            p1 = int(cols[0])
                            p2 = int(cols[1])
                            loops1.append(p1)
                            loops2.append(p2)
                            loops_read += 1
                        except ValueError:
                            pass
                    i += 1
                snapshots_local.append((loops1, loops2, fork1, fork2))
            return snapshots_local

        snapshots = _parse_concatenated_loops(concat_path)
        loop_times = np.arange(len(snapshots)) / float(frames_per_minute)

        left_sum_cat, right_sum_cat, total_sum_cat = [], [], []
        left_uniq_cat, right_uniq_cat, total_uniq_cat = [], [], []
        left_loops_cat, right_loops_cat, total_loops_cat = [], [], []

        for (loops1, loops2, fork1, fork2) in snapshots:
            n_loops = len(loops1)
            if n_loops == 0:
                left_sum_cat.append(0)
                right_sum_cat.append(0)
                total_sum_cat.append(0)
                left_uniq_cat.append(0)
                right_uniq_cat.append(0)
                total_uniq_cat.append(0)
                left_loops_cat.append(0)
                right_loops_cat.append(0)
                total_loops_cat.append(0)
                continue
            right_len = abs(fork2 - fork1)
            current_max_bead = N_parent + right_len - 1 if right_len > 0 else N_parent - 1

            l1, r1, t1 = coverage_sum_of_spans(loops1, loops2, N_parent, current_max_bead, n_loops, fork1, fork2)
            left_sum_cat.append(l1)
            right_sum_cat.append(r1)
            total_sum_cat.append(t1)

            l2, r2, t2 = coverage_unique_sites(loops1, loops2, N_parent, current_max_bead, n_loops, fork1, fork2)
            left_uniq_cat.append(l2)
            right_uniq_cat.append(r2)
            total_uniq_cat.append(t2)

            left_cnt = right_cnt = 0
            for p1, p2 in zip(loops1, loops2):
                cls = _classify_loop_by_daughter(p1, p2, fork1, fork2)
                if cls == "left":
                    left_cnt += 1
                elif cls == "right":
                    right_cnt += 1
            left_loops_cat.append(left_cnt)
            right_loops_cat.append(right_cnt)
            total_loops_cat.append(n_loops)

        series[rep] = {
            "time": loop_times,
            "total_sum": np.array(total_sum_cat),
            "total_uniq": np.array(total_uniq_cat),
            "left_loops": np.array(left_loops_cat),
            "right_loops": np.array(right_loops_cat),
            "total_loops": np.array(total_loops_cat),
        }

    # Make two subplots: coverage + loop counts
    fig, (ax_cov, ax_loops) = plt.subplots(2, 1, figsize=figsize, sharex=True)

    for color, rep in zip(colors, replicates):
        data = series[rep]
        t = data["time"]
        # Coverage: total sum vs total unique, same color, different opacity
        ax_cov.plot(t, data["total_sum"], label=f"{rep} sum-of-spans", color=color, alpha=0.5)
        ax_cov.plot(t, data["total_uniq"], label=f"{rep} unique", color=color, alpha=0.9)

        # Loop counts per daughter: left and right only
        base_rgb = np.array(mcolors.to_rgb(color))
        darker_rgb = tuple(np.clip(base_rgb * 0.7, 0, 1))
        ax_loops.plot(t, data["left_loops"], label=f"{rep} left", color=color, linestyle="-")
        ax_loops.plot(t, data["right_loops"], label=f"{rep} right", color=darker_rgb, linestyle="-")

    ax_cov.set_title("Loop coverage vs time (total daughters)")
    ax_cov.set_ylabel("Coverage (beads)")
    ax_cov.grid(True, alpha=0.3)
    ax_cov.legend()

    ax_loops.set_title("Loop counts per daughter vs time")
    ax_loops.set_xlabel("Time (min)")
    ax_loops.set_ylabel("Number of loops")
    ax_loops.grid(True, alpha=0.3)
    ax_loops.legend()

    plt.tight_layout()
    return fig, (ax_cov, ax_loops)


In [ ]:
# --- 3D sweep runs table (matches analysis/runs.ipynb and submit_jobs_3d_sweep.sh) ---
import pandas as pd
import itertools

# Parameter grid: must match runs.ipynb and submit_jobs_3d_sweep.sh
N_VALUES = [20, 35, 50]
V_VALUES = [200, 350, 500]
TAU_VALUES = [50, 75, 100]
# Run name prefix for 3D sweep (e.g. Feb16 for submit_jobs_3d_sweep)
RUN_DATE = "Mar03"

all_runs = list(itertools.product(N_VALUES, V_VALUES, TAU_VALUES))
runs_list = []
for run_id, (N, v, tau) in enumerate(all_runs, 1):
    S = N * v * tau
    run_name = f"{RUN_DATE}_p{run_id}"
    runs_list.append({"run": run_id, "run_name": run_name, "N": N, "v": v, "tau": tau, "S": S})

runs_df = pd.DataFrame(runs_list)

def filter_runs(runs_df, run_ids=None, N=None, v=None, tau=None):
    """
    Return subset of runs by run ID(s), N, v, and/or tau.
    E.g. filter_runs(runs_df, N=20) for all runs with N=20.
    """
    out = runs_df.copy()
    if run_ids is not None:
        run_ids = [run_ids] if np.isscalar(run_ids) else list(run_ids)
        out = out[out["run"].isin(run_ids)]
    if N is not None:
        out = out[out["N"] == N]
    if v is not None:
        out = out[out["v"] == v]
    if tau is not None:
        out = out[out["tau"] == tau]
    return out.reset_index(drop=True)

# Example: subset with N=20
# filter_runs(runs_df, N=20)
# filter_runs(runs_df, run_ids=[1, 2, 3])
# filter_runs(runs_df, v=350)
runs_df

In [ ]:
runs_df.sort_values("S")

In [ ]:
# Improved loop coverage plotting: left, right, daughters-total, and global-total

def _total_coverage_sum_of_spans(loops1, loops2, N_parent, current_max_bead, num_smc):
    """Total coverage (sum of spans) over all regions (left + mother + right).

    - For anchors < N_parent: use shortest arc on parent circle (0..N_parent-1).
    - For anchors >= N_parent: use linear span (with circular correction if fully replicated).
    - Mixed (cross-daughter) loops are ignored for now.
    """
    total = 0
    right_region_size = max(0, current_max_bead - N_parent + 1)
    is_right_circular = current_max_bead >= 2 * N_parent - 1
    n = min(num_smc, len(loops1), len(loops2))
    for i in range(n):
        p1, p2 = loops1[i], loops2[i]
        if p1 == 0 and p2 == 0:
            continue
        if p1 < N_parent and p2 < N_parent:
            p1c = p1 % N_parent
            p2c = p2 % N_parent
            span = abs(p2c - p1c)
            span = min(span, N_parent - span)
            total += span
        elif p1 >= N_parent and p2 >= N_parent and right_region_size > 0:
            lo = min(p1, p2) - N_parent
            hi = max(p1, p2) - N_parent
            lo = max(0, min(lo, right_region_size - 1))
            hi = max(0, min(hi, right_region_size - 1))
            if hi < lo:
                lo, hi = hi, lo
            span = hi - lo
            if is_right_circular:
                span = min(span, right_region_size - span)
            total += span
    return total


def _total_coverage_unique_sites(loops1, loops2, N_parent, current_max_bead, num_smc):
    """Total coverage (unique sites) over all regions (left + mother + right).

    Uses global boolean arrays:
      - parent_covered[0..N_parent-1]
      - right_covered[0..right_region_size-1]
    """
    parent_covered = [False] * N_parent
    right_region_size = max(0, current_max_bead - N_parent + 1)
    right_covered = [False] * right_region_size if right_region_size > 0 else []
    is_right_circular = current_max_bead >= 2 * N_parent - 1
    n = min(num_smc, len(loops1), len(loops2))
    for i in range(n):
        p1, p2 = loops1[i], loops2[i]
        if p1 == 0 and p2 == 0:
            continue
        if p1 < N_parent and p2 < N_parent:
            p1c = p1 % N_parent
            p2c = p2 % N_parent
            for idx in _beads_on_shorter_arc_circular(p1c, p2c, N_parent):
                parent_covered[idx] = True
        elif p1 >= N_parent and p2 >= N_parent and right_region_size > 0:
            lo = min(p1, p2) - N_parent
            hi = max(p1, p2) - N_parent
            lo = max(0, min(lo, right_region_size - 1))
            hi = max(0, min(hi, right_region_size - 1))
            if hi < lo:
                lo, hi = hi, lo
            if is_right_circular:
                idxs = _beads_on_shorter_arc_circular(lo, hi, right_region_size)
            else:
                idxs = range(lo, hi + 1)
            for idx in idxs:
                if 0 <= idx < right_region_size:
                    right_covered[idx] = True
    total_parent = sum(1 for x in parent_covered if x)
    total_right = sum(1 for x in right_covered if x)
    return total_parent + total_right


def plot_loops(replicate, N_parent=N_PARENT, use_concatenated_loops=True, frames_per_minute=30):
    """Plot loop coverages and counts vs time for a given replicate.

    Plots:
      - Left and right coverage (sum of spans) + two totals:
          * daughters-total (left+right)
          * global-total (left+mother+right)
      - Same for unique-site coverage and overlaps.
      - Loop counts (left, right, total).
    """
    import os
    import numpy as np
    import matplotlib.pyplot as plt

    if use_concatenated_loops:
        concat_path = f"../runs/{replicate}/data/loops/loops_{replicate}.txt"

        def _parse_concatenated_loops(path, N_parent_local=N_parent):
            snapshots_local = []
            with open(path, "r") as f:
                lines = [ln.strip() for ln in f if ln.strip()]
            i = 0
            n = len(lines)
            while i < n:
                line = lines[i]
                if not line.startswith("Number of loops"):
                    i += 1
                    continue
                try:
                    _, val = line.split(":", 1)
                    n_loops = int(val)
                except Exception:
                    n_loops = 0
                i += 1
                if i >= n:
                    break
                fork_line = lines[i]
                try:
                    _, vals = fork_line.split(":", 1)
                    fork_parts = vals.strip().split(",")
                    fork1 = int(fork_parts[0])
                    fork2 = int(fork_parts[1])
                except Exception:
                    fork1 = fork2 = 0
                loops1, loops2 = [], []
                i += 1
                loops_read = 0
                while i < n and loops_read < n_loops and not lines[i].startswith("Number of loops"):
                    cols = lines[i].split()
                    if len(cols) >= 2:
                        try:
                            p1 = int(cols[0])
                            p2 = int(cols[1])
                            loops1.append(p1)
                            loops2.append(p2)
                            loops_read += 1
                        except ValueError:
                            pass
                    i += 1
                snapshots_local.append((loops1, loops2, fork1, fork2))
            return snapshots_local

        snapshots = _parse_concatenated_loops(concat_path)
        loop_times = np.arange(len(snapshots)) / float(frames_per_minute)

        # Coverage and loop counts from concatenated file
        left_sum_cat, right_sum_cat = [], []
        total_daughters_sum_cat, total_global_sum_cat = [], []
        left_uniq_cat, right_uniq_cat = [], []
        total_daughters_uniq_cat, total_global_uniq_cat = [], []
        left_loops_cat, right_loops_cat, total_loops_cat = [], [], []

        for (loops1, loops2, fork1, fork2) in snapshots:
            n_loops = len(loops1)
            if n_loops == 0:
                left_sum_cat.append(0)
                right_sum_cat.append(0)
                total_daughters_sum_cat.append(0)
                total_global_sum_cat.append(0)
                left_uniq_cat.append(0)
                right_uniq_cat.append(0)
                total_daughters_uniq_cat.append(0)
                total_global_uniq_cat.append(0)
                left_loops_cat.append(0)
                right_loops_cat.append(0)
                total_loops_cat.append(0)
                continue
            right_len = abs(fork2 - fork1)
            current_max_bead = N_parent + right_len - 1 if right_len > 0 else N_parent - 1

            # Fork-aware coverage on left/right daughters
            l1, r1, _ = coverage_sum_of_spans(loops1, loops2, N_parent, current_max_bead, n_loops, fork1, fork2)
            left_sum_cat.append(l1)
            right_sum_cat.append(r1)
            total_daughters_sum_cat.append(l1 + r1)

            # Global total coverage (all regions)
            full_sum = _total_coverage_sum_of_spans(loops1, loops2, N_parent, current_max_bead, n_loops)
            total_global_sum_cat.append(full_sum)

            # Unique-site coverage
            l2, r2, _ = coverage_unique_sites(loops1, loops2, N_parent, current_max_bead, n_loops, fork1, fork2)
            left_uniq_cat.append(l2)
            right_uniq_cat.append(r2)
            total_daughters_uniq_cat.append(l2 + r2)

            full_uniq = _total_coverage_unique_sites(loops1, loops2, N_parent, current_max_bead, n_loops)
            total_global_uniq_cat.append(full_uniq)

            # Loop counts per daughter using same classification
            left_cnt = right_cnt = 0
            for p1, p2 in zip(loops1, loops2):
                cls = _classify_loop_by_daughter(p1, p2, fork1, fork2)
                if cls == "left":
                    left_cnt += 1
                elif cls == "right":
                    right_cnt += 1
            left_loops_cat.append(left_cnt)
            right_loops_cat.append(right_cnt)
            total_loops_cat.append(n_loops)

        # Plot coverage and loop counts
        fig, axes = plt.subplots(4, 1, figsize=(10, 11), sharex=False)
        ax0, ax1, ax2, ax3 = axes

        ax0.set_title(f"Coverage (sum of spans) — {replicate}")
        ax0.plot(loop_times, left_sum_cat, label="Left daughter", color="C0", alpha=0.9)
        ax0.plot(loop_times, right_sum_cat, label="Right daughter", color="C1", alpha=0.9)
        ax0.plot(loop_times, total_daughters_sum_cat, label="Total daughters (L+R)", color="C2", linestyle="--", alpha=0.8)
        ax0.plot(loop_times, total_global_sum_cat, label="Global total (L+mother+R)", color="k", linestyle="--", alpha=0.9)
        ax0.set_ylabel("Loop coverage (beads)")
        ax0.legend()
        ax0.grid(True, alpha=0.3)

        ax1.set_title("Coverage (unique sites)")
        ax1.plot(loop_times, left_uniq_cat, label="Left daughter", color="C0", alpha=0.9)
        ax1.plot(loop_times, right_uniq_cat, label="Right daughter", color="C1", alpha=0.9)
        ax1.plot(loop_times, total_daughters_uniq_cat, label="Total daughters (L+R)", color="C2", linestyle="--", alpha=0.8)
        ax1.plot(loop_times, total_global_uniq_cat, label="Global total (L+mother+R)", color="k", linestyle="--", alpha=0.9)
        ax1.set_ylabel("Unique covered sites (beads)")
        ax1.legend()
        ax1.grid(True, alpha=0.3)

        ax2.set_title("Coverage overlaps (sum - unique)")
        ax2.plot(loop_times, np.array(left_sum_cat) - np.array(left_uniq_cat), label="Left daughter", color="C0", alpha=0.9)
        ax2.plot(loop_times, np.array(right_sum_cat) - np.array(right_uniq_cat), label="Right daughter", color="C1", alpha=0.9)
        ax2.plot(loop_times, np.array(total_daughters_sum_cat) - np.array(total_daughters_uniq_cat), label="Total daughters (L+R)", color="C2", linestyle="--", alpha=0.8)
        ax2.plot(loop_times, np.array(total_global_sum_cat) - np.array(total_global_uniq_cat), label="Global total (L+mother+R)", color="k", linestyle="--", alpha=0.9)
        ax2.set_ylabel("Overlaps (beads)")
        ax2.legend()
        ax2.grid(True, alpha=0.3)

        ax3.set_title("Loop counts per daughter")
        ax3.plot(loop_times, left_loops_cat, label="Left daughter loops", color="C0")
        ax3.plot(loop_times, right_loops_cat, label="Right daughter loops", color="C1")
        ax3.plot(loop_times, total_loops_cat, label="Total loops", color="gray", linestyle="--")
        ax3.set_xlabel("Time (min)")
        ax3.set_ylabel("Number of loops")
        ax3.legend()
        ax3.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

In [ ]:
def plot_partitioning_subset(runs_df, run_ids=None, N=None, v=None, tau=None, base_dir=None, partitioning_dir=None, title=None, labels=None, figsize=(8, 5), ylim=None, xlim=None, show_theory=False, C=1.6e-8, frames_per_minute=1):
    """
    Plot partitioning trajectories for a subset of the 48 runs.
    Filter by run_ids (int or list), N, v, and/or tau.
    E.g. plot_partitioning_subset(runs_df, N=20)  # all runs with N=20
         plot_partitioning_subset(runs_df, run_ids=[1, 5, 9], title="...")
    If show_theory=True, overlay theory P = C*N*v*tau*t (same colors as traces, lower opacity).
    """
    subset = filter_runs(runs_df, run_ids=run_ids, N=N, v=v, tau=tau)
    if subset.empty:
        raise ValueError("No runs match the filter")
    replicates = subset["run_name"].tolist()
    if labels is None:
        labels = {row["run_name"]: f"N={row['N']}, v={row['v']}, tau={row['tau']}" for _, row in subset.iterrows()}
    if title is None:
        parts = []
        if N is not None: parts.append(f"N={N}")
        if v is not None: parts.append(f"v={v}")
        if tau is not None: parts.append(f"tau={tau}")
        if run_ids is not None: parts.append(f"run_ids={run_ids}")
        title = "Partitioning vs time — " + (", ".join(parts) if parts else f"{len(replicates)} runs")
    fig, ax = plot_partitioning_trajectories(
        replicates=replicates,
        partitioning_dir=partitioning_dir,
        base_dir=base_dir,
        title=title,
        figsize=figsize,
        labels=labels,
        ylim=ylim,
        xlim=xlim,
        frames_per_minute=frames_per_minute
    )
    if show_theory:
        base_dir = Path(base_dir) if base_dir is not None else Path(".")
        part_dir = Path(partitioning_dir) if partitioning_dir is not None else base_dir / "partitioning"
        first_rep = sorted(replicates)[0]
        minutes, _ = load_partitioning_file(part_dir / f"{first_rep}.txt")
        if len(minutes) > 0:
            colors = plt.cm.tab10(np.linspace(0, 1, max(10, len(replicates))))
            for k, run_name in enumerate(sorted(replicates)):
                row = subset[subset["run_name"] == run_name].iloc[0]
                N, v, tau = row["N"], row["v"], row["tau"]
                P_theory = C * N * v * tau * minutes
                ax.plot(minutes, P_theory, color=colors[k % len(colors)], alpha=0.35, zorder=0)
    return fig, ax



In [ ]:
plot_loops("Mar03_p1", 54338)

In [ ]:
plot_loops("Mar03_p14", 54338)

In [ ]:
def get_partitioning_at_minute(rep_name, minute, part_dir=None, base_dir=None):
    """Load partitioning file for one run and return value at given minute (interpolate if needed)."""
    part_dir = Path(part_dir) if part_dir is not None else Path(base_dir) / "partitioning"
    path = part_dir / f"{rep_name}.txt"
    if not path.exists():
        return np.nan
    minutes, part = load_partitioning_file(path)
    if len(minutes) == 0:
        return np.nan
    if minute in minutes:
        return part[minutes == minute][0]
    # Linear interpolation
    order = np.argsort(minutes)
    minutes = minutes[order]
    part = part[order]
    if minute <= minutes[0]:
        return part[0]
    if minute >= minutes[-1]:
        return part[-1]
    return np.interp(minute, minutes, part)

# Marker options for differentiating N or tau
_MARKERS = ["o", "s", "^", "D", "v", "<", ">", "p", "*", "h", "H", "X"]

def _default_encoding_for_x_axis(x_axis):
    """Sensible defaults for color_by, marker_by, size_by when plotting vs x_axis."""
    params = ["N", "v", "tau"]
    others = [p for p in params if p != x_axis]
    if x_axis in ("S", "L"):
        return {"color_by": "v", "marker_by": "N", "size_by": "tau"}
    if len(others) == 2:
        return {"color_by": others[0], "marker_by": others[1], "size_by": None}
    return {"color_by": others[0], "marker_by": others[1], "size_by": others[2]}

def plot_partitioning_vs(
    runs_df,
    x_axis="S",
    minute=60,
    run_ids=None,
    N=None,
    v=None,
    tau=None,
    base_dir=None,
    partitioning_dir=None,
    ax=None,
    title=None,
    figsize=(10, 6),
    color_by=None,
    marker_by=None,
    size_by=None,
    show_legend=True,
    size_range=(60, 180),
    marker_scale=1.0,
    show_theory=False,
    C=1.6e-8,
    xlim=None,
    xlabel_fontsize=None,
    ylabel_fontsize=None,
    title_fontsize=None,
    tick_labelsize=None,
    legend_fontsize=None,
):
    """
    Plot extent of partitioning at a given minute vs N, v, tau, S, or loop coverage.

    x_axis: "N", "v", "tau", "S", or "L" — quantity on the x-axis.
        When x_axis="L", the x values are S / 543379 (loop coverage), where S = N * v * tau.
    Use color_by, marker_by, size_by (each "N", "v", or "tau"; or None) to distinguish parameters.
    Defaults: if not set, chosen from the other dimensions (e.g. when x_axis="v": color_by="tau", marker_by="N").
    size_range = (min_s, max_s) for size_by. marker_scale multiplies all marker sizes (default 1.0).
    If show_theory is True:
        - when x_axis=="S": overlay theory P = C * S * t as a black line.
        - when x_axis=="L": overlay theory P = C * L * t as a black line (L = loop coverage).
    """
    if x_axis not in ("N", "v", "tau", "S", "L"):
        raise ValueError("x_axis must be one of 'N', 'v', 'tau', 'S', 'L'")
    defaults = _default_encoding_for_x_axis(x_axis)
    if color_by is None:
        color_by = defaults["color_by"]
    if marker_by is None:
        marker_by = defaults["marker_by"]
    if size_by is None:
        size_by = defaults["size_by"]

    subset = filter_runs(runs_df, run_ids=run_ids, N=N, v=v, tau=tau)
    if subset.empty:
        raise ValueError("No runs match the filter")
    part_dir = Path(partitioning_dir) if partitioning_dir is not None else Path(base_dir) / "partitioning"
    part_at_t = []
    for _, row in subset.iterrows():
        p = get_partitioning_at_minute(row["run_name"], minute, part_dir=part_dir, base_dir=base_dir)
        part_at_t.append(p)
    subset = subset.copy()
    subset["partitioning"] = part_at_t
    subset = subset.dropna(subset=["partitioning"])
    if subset.empty:
        raise FileNotFoundError(f"No partitioning data at minute {minute} for the selected runs in {part_dir}")

    # Choose x column and label, including loop coverage option (L = S / 543379)
    if x_axis == "L":
        if "S" not in subset.columns:
            subset = subset.copy()
            subset["S"] = subset["N"] * subset["v"] * subset["tau"]
        subset = subset.copy()
        subset["L"] = subset["S"] / 543379.0
        x_col = "L"
        x_label = "Loop Coverage"
    else:
        x_col = x_axis
        x_label = "S (N × v × tau)" if x_axis == "S" else x_axis

    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
        created_fig = True
    else:
        fig, created_fig = None, False

    color_vals = sorted(subset[color_by].unique()) if color_by and color_by in subset.columns else [None]
    marker_vals = sorted(subset[marker_by].unique()) if (marker_by and marker_by in subset.columns) else [None]
    size_vals = sorted(subset[size_by].unique()) if (size_by and size_by in subset.columns) else [None]
    colors = plt.cm.tab10(np.linspace(0, 1, max(len(color_vals), 1)))
    markers = [_MARKERS[i % len(_MARKERS)] for i in range(len(marker_vals))]
    s_min, s_max = size_range
    sizes = np.linspace(s_min, s_max, len(size_vals)) if size_vals and size_vals[0] is not None else [60]
    if show_theory:
        if x_axis == "S":
            x_min = min(0.0, float(subset[x_col].min()))
            x_max = float(subset[x_col].max())
            x_theory = np.linspace(x_min, x_max, 200)
            P_theory = C * minute * x_theory
            ax.plot(x_theory, P_theory, color="0.5", linestyle="-", alpha=0.8, zorder=0)
        elif x_axis == "L":
            x_min = min(0.0, float(subset[x_col].min()))
            x_max = float(subset[x_col].max())
            x_theory = np.linspace(x_min, x_max, 200)
            P_theory = C * minute * x_theory
            ax.plot(x_theory, P_theory, color="0.5", linestyle="-", alpha=0.8, zorder=0)
    for ci, cval in enumerate(color_vals):
        for mi, mval in enumerate(marker_vals):
            for si, sval in enumerate(size_vals):
                mask = (subset[color_by] == cval) if (color_by and cval is not None) else np.ones(len(subset), dtype=bool)
                if marker_by and mval is not None:
                    mask = mask & (subset[marker_by] == mval)
                if size_by and sval is not None:
                    mask = mask & (subset[size_by] == sval)
                if not mask.any():
                    continue
                lbl = []
                if color_by and cval is not None:
                    lbl.append(f"{color_by}={cval}")
                if marker_by and mval is not None:
                    lbl.append(f"{marker_by}={mval}")
                if size_by and sval is not None:
                    lbl.append(f"{size_by}={sval}")
                label = ", ".join(lbl) if lbl else None
                pt_size = int(sizes[si]) if (size_by and sval is not None) else int(default_s)
                ax.scatter(
                    subset.loc[mask, x_col], subset.loc[mask, "partitioning"],
                    c=[colors[ci]] if (color_by and cval is not None) else None,
                    marker=markers[mi] if (marker_by and mval is not None) else "o",
                    s=pt_size, label=label, alpha=0.85, edgecolors="k", linewidths=0.5,
                )
    if show_legend:
        leg = ax.legend(loc='lower right')
        if leg is not None and legend_fontsize is not None:
            for text in leg.get_texts():
                text.set_fontsize(legend_fontsize)
    elif len(color_vals) == 1 and color_vals[0] is None and (len(marker_vals) == 1 and marker_vals[0] is None) and (len(size_vals) == 1 and size_vals[0] is None):
        ax.scatter(subset[x_col], subset["partitioning"], alpha=0.8, s=int(50 * marker_scale))

    ax.set_xlabel(x_label, fontsize=xlabel_fontsize)
    ax.set_ylabel(f"Partitioning at {minute} min", fontsize=ylabel_fontsize)
    if tick_labelsize is not None:
        ax.tick_params(axis="both", labelsize=tick_labelsize)
    if title is None:
        title = f"Partitioning at {minute} min vs {x_label}"
    ax.set_title(title, fontsize=title_fontsize)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0.0, 1.0)
    if xlim is not None:
        xmin = xlim[0] if xlim[0] is not None else None
        xmax = xlim[1] if len(xlim) > 1 and xlim[1] is not None else None
        ax.set_xlim(left=xmin, right=xmax)
    if created_fig:
        plt.tight_layout()
    return (fig, ax) if created_fig else (None, ax)

def plot_partitioning_vs_S(runs_df, minute=60, run_ids=None, N=None, v=None, tau=None, base_dir=None, partitioning_dir=None, ax=None, title="Extent of partitioning at 60 s vs S", figsize=(10, 6), color_by="v", marker_by="N", size_by="tau", show_legend=True, size_range=(60, 180), marker_scale=1.0, show_theory=False, C=1.6e-8):
    """
    Plot extent of partitioning at a given minute (default 60 s) vs S = N*v*tau.
    Differentiate: color_by, marker_by, size_by (each \"v\", \"N\", or \"tau\", or None). size_range = (min_s, max_s) for size_by.
    If show_theory=True, overlay theory P = C*S*t as a black line.
    """
    return plot_partitioning_vs(runs_df, x_axis="S", minute=minute, run_ids=run_ids, N=N, v=v, tau=tau,
        base_dir=base_dir, partitioning_dir=partitioning_dir, ax=ax, title=title, figsize=figsize,
        color_by=color_by, marker_by=marker_by, size_by=size_by, show_legend=show_legend, size_range=size_range,
        marker_scale=marker_scale, show_theory=show_theory, C=C)

In [ ]:
# --- Examples: plot subset by N, v, or tau; and partitioning at 60 s vs S ---

# Trajectories for all runs with X
fig, ax = plot_partitioning_subset(runs_df, base_dir=base_dir, ylim=(0.0, 1.0), 
show_theory=False, C=1.7e-8, xlim=(0,100), frames_per_minute=1)
ax.get_legend().remove()
plt.show()


**Partitioning vs N, v, or tau** — Use `plot_partitioning_vs(runs_df, x_axis="N"|"v"|"tau"|"S", minute=..., ...)`. Same encoding as vs S: e.g. when plotting vs v, use `color_by="tau"`, `marker_by="N"` to distinguish parameters (defaults are set automatically).

In [ ]:
# Toggle x_axis to plot partitioning (at given time) vs N, v, tau, or S.
# Defaults: shapes = N, colors = other param (e.g. tau when x_axis="v"). Override with color_by=, marker_by=, size_by=.
minute = 40
x_axis = "tau"   # "N", "v", "tau", or "S"
fig, ax = plot_partitioning_vs(runs_df, x_axis=x_axis, minute=minute, base_dir=base_dir,
    color_by="v", marker_by="N", title=f"Partitioning at {minute} min vs {x_axis}")
plt.show()

In [ ]:

# Extent of partitioning at 60 s vs S (points colored by v) "LONG TIME"
fig, ax = plot_partitioning_vs(runs_df, x_axis="L", minute=60, base_dir=base_dir, title="", 
size_by="tau",color_by="v",marker_by="N",show_legend=False, show_theory=True, C=0.01,xlim=(0,4),
    xlabel_fontsize=14,
    ylabel_fontsize=14,
    title_fontsize=14,
    tick_labelsize=14,
    legend_fontsize=12,figsize=(6, 5),
    marker_scale=9)
    
# Shade region above 0.8 in light green
ymin, ymax = ax.get_ylim()
#ax.axhspan(0.8, ymax, color="green", alpha=0.1, zorder=0)

# Add horizontal line at 0.8
ax.axhline(0.703, color="green", linestyle="--", linewidth=1.5)

fig.savefig("loop_coverage_vs_partitioning_60min.png", dpi=600, bbox_inches="tight")
plt.show()

In [ ]:
# Extent of partitioning at 60 s vs S (points colored by v) "LONG TIME"
fig, ax = plot_partitioning_vs(runs_df, minute=12, x_axis="L", base_dir=base_dir, title="", 
size_by="tau",color_by="v",marker_by="N",show_legend=False, show_theory=True, C=0.006,xlim=(0,4),
    xlabel_fontsize=14,
    ylabel_fontsize=14,
    title_fontsize=16,
    tick_labelsize=12,
    legend_fontsize=12)
plt.show()

In [ ]:
plot_partitioning_trajectories(
    dates=["Mar06"],
    base_dir=base_dir,
    title="Partitioning",
    labels={
    "Mar06_1": "N50_v100_t200","Mar06_2": "N5_v500_t400" 
},
    ylim=(0.0, 1.0)
)
plt.show()

In [ ]:
plot_loops("Mar06_1", 54338)

In [ ]:
plot_loops("Mar06_2", 54338)

In [ ]:
plot_partitioning_trajectories(
    dates=["Mar03_p1_lammpstrj", "Mar03_p5_lammpstrj"],
    base_dir=base_dir,
    title="Partitioning",
    labels={
    "Mar03_p1_lammpstrj": "N20_v350_t75", 
    "Mar03_p5_lammpstrj": "N20_v350_t75", 
},
    ylim=(0.0, 1.0)
)
plt.show()

In [ ]:
plot_partitioning_trajectories(
    dates=["Mar07_1", "Mar03_p9"],
    base_dir=base_dir,
    title="Partitioning",
    labels={
    "Mar07_1": "no topo",
    "Mar03_p9": "with topo"
},
    ylim=(0.0, 1.0)
)
plt.show()

In [ ]:
fig, ax = plot_partitioning_trajectories(
    dates=["Feb03_1", "Feb04_1"],
    base_dir=base_dir,
    title="Effect of topological pre-stress on partitioning",
    labels={
    "Feb03_1": "15 minutes topological pre-stress",
    "Feb04_1": "No topological pre-stress"
},
    ylim=(0.0, 1.0), frames_per_minute=1
)

# Adjust font sizes
ax.tick_params(axis="both", labelsize=14)
ax.set_xlabel(ax.get_xlabel(), fontsize=14)
ax.set_ylabel(ax.get_ylabel(), fontsize=14)
ax.set_title(ax.get_title(), fontsize=14)

# X ticks every 15 minutes starting at -15
xmin, xmax = ax.get_xlim()
xticks = np.arange(-15, xmax + 1e-6, 15)
ax.set_xticks(xticks)
ax.set_xticklabels([f"{int(t)}" for t in xticks])

# Make legend labels larger
legend = ax.get_legend()
if legend is not None:
    legend.set_title(legend.get_title().get_text(),
                     prop={"size": 16})  # legend title size (optional)
    for text in legend.get_texts():
        text.set_fontsize(14)          # legend label size

# Shift all traces so they start at -15 instead of 0
for line in ax.lines:
    x = line.get_xdata()
    line.set_xdata(x - 15)

# Optionally update x limits to keep everything visible
xmin, xmax = ax.get_xlim()
ax.set_xlim(-15, 62)
plt.show()


In [ ]:
plot_partitioning_trajectories(
    dates=["Feb16_p12", "Mar03_p9", "Feb06_1", "Jan26_3", "Feb03_1", "Feb03_2", "Jan29_1"],
    base_dir=base_dir,
    title="Partitioning",
    labels={
    "Feb16_p12" : "regular no swell",
    "Mar03_p9": "topo regular stalling 25 s replication fork",
    "Feb06_1": "topo regular fall off immediately (0s dwell)",
    "Jan26_3": "topo regular complete stalling at replication fork (100s dwell)",
    "Feb03_1": "topo after",
    "Feb03_2": "no topo",
    "Jan29_1": "SMC blocking (20s)"
},
    ylim=(0.0, 1.0), xlim=(0,60)
)
plt.show()

In [ ]:
# BLOCKING CHOICES RUNS
plot_partitioning_trajectories(
    dates=["Mar09_1_lammpstrj", "Mar03_p9_lammpstrj", "Mar08_1_lammpstrj"],
    base_dir=base_dir,
    title="Partitioning",
    labels={
    "Mar09_1_lammpstrj" : "tau 100s, tau_SMC 20s, and tau_RF 100s",
    "Mar03_p9_lammpstrj": "tau 100s, tau_SMC 0s, and tau_RF 25s",
    "Mar08_1_lammpstrj": "tau 100s, tau_SMC 0s, and tau_RF 100s"
},
    ylim=(0.0, 1.0), xlim=(0,90),
    frames_per_minute=15.0,
    xlabel_fontsize=14,
    ylabel_fontsize=14,
    title_fontsize=14,
    tick_labelsize=14,
    legend_fontsize=12,
)
plt.show()

In [ ]:
# LOOP COVERAGE RUNS
fig, ax = plot_partitioning_trajectories(
    dates=["Mar03_p1_lammpstrj", 
    "Mar03_p4_lammpstrj", 
    "Mar03_p13_lammpstrj", 
    "Mar03_p12_lammpstrj", 
    "Mar03_p10_lammpstrj",
    "Mar03_p9_lammpstrj", 
    "Mar03_p12_lammpstrj", 
    "Mar03_p18_lammpstrj", 
    "Mar03_p6_lammpstrj", 
    "Mar03_p21_lammpstrj"],
    base_dir=base_dir,
    title="",
    labels={
    "Mar03_p1_lammpstrj":"0.37 (20/200/50)", 
    "Mar03_p10_lammpstrj":"0.64 (35/200/50)",
    "Mar03_p4_lammpstrj":"0.64 (20/350/50)", 
    #"Mar03_p13_lammpstrj":"1.12 (35/350/50)",
    "Mar03_p12_lammpstrj":"1.29 (35/200/100)",
    "Mar03_p6_lammpstrj":"1.29 (20/350/100)", 
    #"Mar03_p15_lammpstrj":"35/350/100",
    #"Mar03_p16_lammpstrj":"1.61 (35/500/50)",
    "Mar03_p9_lammpstrj":"1.84 (20/500/100)", 
    "Mar03_p21_lammpstrj":"1.84 (50/200/100)", 
    #"Mar03_p18_lammpstrj":"3.22 (35/500/100)", 
    #"Mar03_p24_lammpstrj":"50/350/100", 0.64
},
    ylim=(0.0, 1.0), xlim=(0,80),
    frames_per_minute=15.0,
    xlabel_fontsize=14,
    ylabel_fontsize=14,
    title_fontsize=14,
    tick_labelsize=14,
    legend_fontsize=12,
)
fig.savefig("partitioning.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
fig, (ax_cov, ax_loops) = plot_loops_multiple(
    ["Mar03_p9", "Mar06_1", "Mar06_2"],
    N_parent=N_PARENT,          # same as in plot_loops
    use_concatenated_loops=True,
    frames_per_minute=30,
    figsize=(10, 6),
)

In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt

def style_loops_multiple(
    fig,
    ax_cov,
    ax_loops,
    figsize=(8, 5),
    # font sizes
    title_size=16,
    label_size=14,
    tick_size=12,
    legend_size=12,
    # legend on/off
    show_legend=True,
    # titles / axis labels (None = keep existing)
    cov_title=None,
    loops_title=None,
    cov_xlabel=None,
    cov_ylabel=None,
    loops_xlabel=None,
    loops_ylabel=None,
    # coverage y-axis scaling
    cov_y_in_mb=False,
    mb_factor=100000.0,
    # axis limits (None = keep current)
    cov_xlim=None,
    cov_ylim=None,
    loops_xlim=None,
    loops_ylim=None,
    show=True,
):
    """Adjust figure size, fonts, labels, legends, limits, and optional MB scaling for plot_loops_multiple."""
    fig.set_size_inches(*figsize)

    # Optional: convert coverage y-axis to MB (run once)
    if cov_y_in_mb and not getattr(ax_cov, "_y_scaled_to_mb", False):
        for line in ax_cov.get_lines():
            y = line.get_ydata()
            line.set_ydata(y / mb_factor)
        ax_cov._y_scaled_to_mb = True  # mark scaled

    # Titles and labels
    if cov_title is not None:
        ax_cov.set_title(cov_title)
    if loops_title is not None:
        ax_loops.set_title(loops_title)

    if cov_xlabel is not None:
        ax_cov.set_xlabel(cov_xlabel)
    if cov_ylabel is not None:
        ax_cov.set_ylabel(cov_ylabel)

    if loops_xlabel is not None:
        ax_loops.set_xlabel(loops_xlabel)
    if loops_ylabel is not None:
        ax_loops.set_ylabel(loops_ylabel)

    # If we scaled to MB and no explicit cov_ylabel was given, update label text
    if cov_y_in_mb and cov_ylabel is None:
        ax_cov.set_ylabel("Coverage (MB)")

    # Axis limits
    if cov_xlim is not None:
        ax_cov.set_xlim(*cov_xlim)
    if cov_ylim is not None:
        ax_cov.set_ylim(*cov_ylim)
    if loops_xlim is not None:
        ax_loops.set_xlim(*loops_xlim)
    if loops_ylim is not None:
        ax_loops.set_ylim(*loops_ylim)

    # Font sizes + legends
    for ax in (ax_cov, ax_loops):
        ax.title.set_fontsize(title_size)
        ax.xaxis.label.set_fontsize(label_size)
        ax.yaxis.label.set_fontsize(label_size)
        ax.tick_params(axis="both", labelsize=tick_size)

        leg = ax.get_legend()
        if leg is not None:
            if show_legend:
                for txt in leg.get_texts():
                    txt.set_fontsize(legend_size)
                leg.set_visible(True)
            else:
                leg.set_visible(False)

    fig.tight_layout()

    if show:
        display(fig)


In [ ]:
style_loops_multiple(
    fig,
    ax_cov,
    ax_loops,
    figsize=(8, 5),    # tweak without recomputing data
    title_size=14,
    label_size=14,
    tick_size=14,
    legend_size=0,
    show_legend=False,
    cov_title="",                   # remove coverage title
    loops_title="",    # custom loops title
    cov_ylabel="Coverage (Mb)",     # or let helper set this when cov_y_in_mb=True
    loops_ylabel="Loop count",
    cov_y_in_mb=True, 
    cov_xlim=(0, 60),
    cov_ylim=(0, 3.0),
    loops_xlim=(0, 60),
    loops_ylim=(0, 70),
)

In [ ]:
# EXTREME PARAMETER RUNS
fig, ax = plot_partitioning_trajectories(
    dates=[
    "Mar03_p9_lammpstrj", 
    "Mar03_p21_lammpstrj",
    "Mar06_1_lammpstrj", 
    "Mar06_2_lammpstrj" 
],
    base_dir=base_dir,
    title="",
    labels={
    "Mar03_p9_lammpstrj":"1.84 (20/500/100)", 
    "Mar03_p21_lammpstrj":"1.84 (50/200/100)", 
    "Mar06_1_lammpstrj":"1.84 (50/100/200)", 
    "Mar06_2_lammpstrj":"1.84 (5/500/400)", 
},
    ylim=(0.0, 1.0), xlim=(0,60),
    frames_per_minute=15.0,
    xlabel_fontsize=14,
    ylabel_fontsize=14,
    title_fontsize=14,
    tick_labelsize=14,
    legend_fontsize=14,
)
fig.set_size_inches((6,5))
fig.savefig("partitioning_extreme_parameters.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# TOPO AND SWELL CHOICES RUNS
fig, ax = plot_partitioning_trajectories(
    dates=["Mar03_p9_lammpstrj","Feb16_p12_lammpstrj", "Feb03_1_lammpstrj", "Feb03_2_lammpstrj"],
    base_dir=base_dir,
    title="",
    labels={
    "Mar03_p9_lammpstrj":"regular",
    "Feb16_p12_lammpstrj" : "regular no swell",
    "Feb03_1_lammpstrj": "topo after",
    "Feb03_2_lammpstrj": "no topo",
},
    ylim=(0.0, 1.0), xlim=(0,60),
    frames_per_minute=15.0,
    xlabel_fontsize=14,
    ylabel_fontsize=14,
    title_fontsize=14,
    tick_labelsize=14,
    legend_fontsize=14,
)
fig.set_size_inches((6,5))
fig.savefig("partitioning_toposwell_parameters.png", dpi=300, bbox_inches="tight")
plt.show()